# Notebook setup
This is the notebook for homework: batch-processing.
The first section includes importing libaries, downloading data, and start a spark session.
The full homework can be found in this [link](https://github.com/Vijak-k/data-engineering-zoomcamp/tree/main/06-batch-processing/homework).

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
! wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 16:28:53--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.172.0.195, 18.172.0.48, 18.172.0.12, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.172.0.195|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: 'yellow_tripdata_2025-11.parquet'

     0K .......... .......... .......... .......... ..........  0% 7.07M 10s
    50K .......... .......... .......... .......... ..........  0% 7.76M 9s
   100K .......... .......... .......... .......... ..........  0% 16.7M 7s
   150K .......... .......... .......... .......... ..........  0% 14.4M 7s
   200K .......... .......... .......... .......... ..........  0% 18.7M 6s
   250K .......... .......... .......... .......... ..........  0% 19.9M 6s
   300K .......... .......... .......... .......... ..........  0% 50.1M 

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

# Question 1

In [ ]:
# Run the following command to see the output
spark.version

'4.1.1'

# Question 2
Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [4]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [6]:
df.rdd.getNumPartitions()

16

In [10]:
df.repartition(4).write.parquet('yellow_tripdata_2025-11')

In [11]:
import os

path = 'yellow_tripdata_2025-11'
# Filter for actual parquet data files, excluding metadata like _SUCCESS
files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.parquet')]

if files:
    # Get sizes in bytes, convert to MB (1024 * 1024)
    sizes = [os.path.getsize(f) / (1024 * 1024) for f in files]
    print(f"partition sizes in MB {sizes}")
    avg_size = sum(sizes) / len(sizes)
    print(f"Average Parquet file size: {avg_size:.2f} MB")
else:
    print("No Parquet files found in the directory.")

partition sizes in MB [24.403146743774414, 24.404020309448242, 24.424675941467285, 24.432442665100098]
Average Parquet file size: 24.42 MB


# Question 3
How many taxi trips were there on the 15th of November?
Consider only trips that started on the 15th of November.

In [13]:
df_4 = spark.read.parquet('yellow_tripdata_2025-11/*')

In [14]:
df_4.registerTempTable('trips_data')

d:\Project\data-engineering-zoomcamp\.venv\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [19]:
spark.sql("""
SELECT
    count(*) AS count
FROM
    trips_data
WHERE
    tpep_pickup_datetime >= '2025-11-15 00:00:00' AND
    tpep_pickup_datetime < '2025-11-16 00:00:00'
""").show()

+------+
| count|
+------+
|162604|
+------+



In [22]:
df_4.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



# Question 4
What is the length of the longest trip in the dataset in hours?

In [33]:
spark.sql("""
SELECT
    tpep_pickup_datetime AS pickup_datetime,
    tpep_dropoff_datetime AS dropoff_datetime,
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0 AS duration_hours
FROM
    trips_data
ORDER BY
    duration_hours DESC
LIMIT
    1
""").show()

+-------------------+-------------------+--------------+
|    pickup_datetime|   dropoff_datetime|duration_hours|
+-------------------+-------------------+--------------+
|2025-11-26 20:22:12|2025-11-30 15:01:00|     90.646667|
+-------------------+-------------------+--------------+



# Question 5
What is the length of the longest trip in the dataset in hours?
Answer: 4040

# Question 6
Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

If multiple answers are correct, select any

In [34]:
! wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 21:59:54--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.172.0.195, 18.172.0.77, 18.172.0.48, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.172.0.195|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: 'taxi_zone_lookup.csv'

     0K .......... ..                                         100% 50.7M=0s

2026-03-09 21:59:55 (50.7 MB/s) - 'taxi_zone_lookup.csv' saved [12331/12331]



In [35]:
df_zone = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [36]:
df_join = df_4.join(df_zone, df_4.PULocationID == df_zone.LocationID)

In [38]:
df_join.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+

In [37]:
df_join.registerTempTable('trips_data_with_zones')

d:\Project\data-engineering-zoomcamp\.venv\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [ ]:
spark.sql("""
SELECT
    Zone,
    COUNT(*) AS count
FROM
    trips_data_with_zones
GROUP BY
    ZONE
ORDER BY
    2 ASC
LIMIT
    10
""").show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
|   Rossville/Woodrow|    4|
|         Great Kills|    4|
| Green-Wood Cemetery|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
+--------------------+-----+

